# Model Testing & Evaluation

This notebook contains working code to test, score, and evaluate your Isolation Forest + DIFFI anomaly detection model.

## Step 1: Setup & Data Preparation

In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

In [ ]:
# Load and prepare data
df = pd.read_csv('4shark_surveys_species_count.csv')

# Transpose so surveys are rows, species are columns
df_T = df.set_index("species_name").T
df_T.index.name = None
df_T = df_T.reset_index(drop=True)
df_T = df_T.rename_axis(None, axis=1)

# Select top 50 species by prevalence
species_counts = df_T.sum(axis=0).sort_values(ascending=False)
top_species = species_counts.head(50).index.tolist()
df_model = df_T[top_species].copy()

print(f"Data loaded and prepared:")
print(f"  Total samples: {len(df_model)}")
print(f"  Features used: {len(top_species)} (top species)")
print(f"  Total species in dataset: {df_T.shape[1]}")

In [ ]:
# Split into train/test
df_train = df_model.sample(frac=0.85, random_state=42)
df_test = df_model.drop(df_train.index)

print(f"Train set: {len(df_train)} samples")
print(f"Test set: {len(df_test)} samples")

## Step 2: Train Model

In [ ]:
# Train Isolation Forest
IF = IsolationForest(
    contamination=0.1,  # Expect ~10% anomalies
    random_state=42,
    n_estimators=100,
    max_features=1.0,
    max_samples='auto'
)

IF.fit(df_train)

print("Model trained successfully!")
print(f"Trees: {IF.n_estimators}")
print(f"Features: {IF.n_features_in_}")
print(f"Samples per tree: {IF.max_samples_}")

## Step 3: Test Predictions

In [ ]:
# Get predictions on test set
test_predictions = IF.predict(df_test)  # -1 for anomaly, 1 for normal
test_scores = IF.score_samples(df_test)  # Anomaly scores

# Count results
n_normal = (test_predictions == 1).sum()
n_anomalies = (test_predictions == -1).sum()

print("\n" + "="*60)
print("TEST SET PREDICTION RESULTS")
print("="*60)
print(f"Normal Samples: {n_normal} ({100*n_normal/len(test_predictions):.1f}%)")
print(f"Anomalies: {n_anomalies} ({100*n_anomalies/len(test_predictions):.1f}%)")
print("="*60)

## Step 4: Score Statistics

In [ ]:
# Separate scores by prediction
normal_scores = test_scores[test_predictions == 1]
anomaly_scores = test_scores[test_predictions == -1]

print("\n" + "="*60)
print("ANOMALY SCORE STATISTICS")
print("="*60)

print(f"\nAll Scores:")
print(f"  Mean: {test_scores.mean():.4f}")
print(f"  Std: {test_scores.std():.4f}")
print(f"  Range: [{test_scores.min():.4f}, {test_scores.max():.4f}]")
print(f"  Median: {np.median(test_scores):.4f}")

print(f"\nNormal Samples:")
print(f"  Count: {len(normal_scores)}")
print(f"  Mean: {normal_scores.mean():.4f}")
print(f"  Std: {normal_scores.std():.4f}")
print(f"  Range: [{normal_scores.min():.4f}, {normal_scores.max():.4f}]")

if len(anomaly_scores) > 0:
    print(f"\nAnomalies:")
    print(f"  Count: {len(anomaly_scores)}")
    print(f"  Mean: {anomaly_scores.mean():.4f}")
    print(f"  Std: {anomaly_scores.std():.4f}")
    print(f"  Range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
    print(f"\n  Score Separation: {normal_scores.mean() - anomaly_scores.mean():.4f}")
    if abs(normal_scores.mean() - anomaly_scores.mean()) > 0.05:
        print("  ✓ GOOD: Clear separation between normal and anomalies")
    else:
        print("  ⚠ WEAK: Subtle separation between normal and anomalies")
else:
    print(f"\nAnomalies: NONE DETECTED")
    print("  ✓ All test samples appear normal relative to training data")

print("="*60)

## Step 5: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Histogram
axes[0, 0].hist(normal_scores, bins=12, alpha=0.7, label='Normal', color='green', edgecolor='black')
if len(anomaly_scores) > 0:
    axes[0, 0].hist(anomaly_scores, bins=8, alpha=0.7, label='Anomaly', color='red', edgecolor='black')
axes[0, 0].set_xlabel('Anomaly Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Score Distribution: Normal vs Anomalies')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(alpha=0.3)

# 2. Box plot
if len(anomaly_scores) > 0:
    bp = axes[0, 1].boxplot([normal_scores, anomaly_scores], 
                              labels=['Normal', 'Anomaly'],
                              patch_artist=True, 
                              notch=True)
    for patch, color in zip(bp['boxes'], ['lightgreen', 'lightcoral']):
        patch.set_facecolor(color)
else:
    bp = axes[0, 1].boxplot([normal_scores], labels=['Normal'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgreen')
    
axes[0, 1].set_ylabel('Anomaly Score')
axes[0, 1].set_title('Score Comparison')
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Classification pie
sizes = [n_normal, n_anomalies]
labels = [f'Normal\n({n_normal})', f'Anomalies\n({n_anomalies})']
colors = ['green', 'red']
explode = (0, 0.1) if n_anomalies > 0 else (0,)
axes[1, 0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', 
                explode=explode, startangle=90)
axes[1, 0].set_title('Test Set Classification')

# 4. Summary statistics
summary_text = f"""MODEL EVALUATION SUMMARY

Dataset:
  Train Samples: {len(df_train)}
  Test Samples: {len(df_test)}
  Features: {len(top_species)} species

Model Config:
  Algorithm: Isolation Forest
  Trees: 100
  Contamination: 0.1 (10%)

Results:
  Normal: {n_normal} ({100*n_normal/len(test_predictions):.1f}%)
  Anomalies: {n_anomalies} ({100*n_anomalies/len(test_predictions):.1f}%)
  
Score Stats:
  Normal Mean: {normal_scores.mean():.4f}
  Normal Std: {normal_scores.std():.4f}
  Anomaly Mean: {anomaly_scores.mean():.4f} (if any)
  Separation: {abs(normal_scores.mean() - anomaly_scores.mean()):.4f}

Quality Assessment:
  {'✓ GOOD' if abs(normal_scores.mean() - anomaly_scores.mean()) > 0.05 else '✓ NORMAL' if len(anomaly_scores) == 0 else '⚠ FAIR'}
"""

axes[1, 1].text(0.05, 0.95, summary_text, 
                 transform=axes[1, 1].transAxes,
                 fontsize=10, verticalalignment='top',
                 family='monospace',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[1, 1].axis('off')

plt.suptitle('Model Evaluation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Visualization complete!")

## Step 6: Detailed Anomaly Analysis

In [ ]:
if len(anomaly_scores) > 0:
    print("\n" + "="*80)
    print("DETECTED ANOMALIES - DETAILED ANALYSIS")
    print("="*80)
    
    # Sort anomalies by score
    anomaly_indices = np.where(test_predictions == -1)[0]
    anomaly_idx_sorted = anomaly_indices[np.argsort(test_scores[anomaly_indices])]
    
    for rank, idx in enumerate(anomaly_idx_sorted[:5], 1):
        print(f"\n#{rank} ANOMALY (Index: {idx}, Score: {test_scores[idx]:.4f})")
        print("-" * 80)
        
        # Top species in this survey
        survey_species = df_test.iloc[idx].sort_values(ascending=False).head(10)
        print("Top Species in this Survey:")
        for sp_name, count in survey_species.items():
            print(f"  {sp_name}: {int(count)}")
        
        # Compare to averages
        print("\nComparison to Normal Surveys:")
        normal_means = df_test[test_predictions == 1].mean()
        for sp_name, count in survey_species.head(5).items():
            normal_avg = normal_means[sp_name]
            diff_pct = 100 * (count - normal_avg) / (normal_avg + 1)
            print(f"  {sp_name}: {int(count)} (vs avg {normal_avg:.1f}, {diff_pct:+.0f}%)")
else:
    print("No anomalies detected in test set.")

## Step 7: Model Quality Assessment

In [ ]:
print("\n" + "="*80)
print("MODEL QUALITY ASSESSMENT")
print("="*80)

# Check 1: Detection rate
detection_rate = 100 * n_anomalies / len(test_predictions)
print(f"\n1. ANOMALY DETECTION RATE: {detection_rate:.1f}%")
if 8 <= detection_rate <= 12:
    print("   ✓ GOOD: Close to expected 10%")
elif 5 <= detection_rate <= 15:
    print("   ⚠ FAIR: Within acceptable range (5-15%)")
elif detection_rate < 5:
    print("   ✗ LOW: Consider increasing contamination parameter")
else:
    print("   ✗ HIGH: Consider decreasing contamination parameter")

# Check 2: Score separation
if len(anomaly_scores) > 0:
    separation = normal_scores.mean() - anomaly_scores.mean()
    print(f"\n2. SCORE SEPARATION: {separation:.4f}")
    if separation > 0.05:
        print("   ✓ GOOD: Clear distinction between normal and anomalies")
    elif separation > 0.01:
        print("   ⚠ FAIR: Moderate distinction")
    else:
        print("   ✗ POOR: Scores too similar, model may not be discriminative")
else:
    print(f"\n2. SCORE SEPARATION: N/A (no anomalies detected)")
    print("   ✓ All test data appears normal - clean baseline")

# Check 3: Score distribution
print(f"\n3. SCORE DISTRIBUTION:")
print(f"   Normal Score Std: {normal_scores.std():.4f}")
if normal_scores.std() < 0.15:
    print("   ✓ GOOD: Normal scores tightly clustered")
else:
    print("   ⚠ Note: Normal scores somewhat spread out")

if len(anomaly_scores) > 0:
    print(f"   Anomaly Score Std: {anomaly_scores.std():.4f}")

# Check 4: Overall assessment
print(f"\n4. OVERALL MODEL STATUS:")
quality_score = 0
if 8 <= detection_rate <= 12:
    quality_score += 1
if len(anomaly_scores) > 0 and separation > 0.05:
    quality_score += 1
elif len(anomaly_scores) == 0:
    quality_score += 1
if normal_scores.std() < 0.15:
    quality_score += 1

if quality_score >= 3:
    print("   ✓✓✓ EXCELLENT: Model is performing well")
elif quality_score >= 2:
    print("   ✓✓ GOOD: Model performance is acceptable")
else:
    print("   ✓ FAIR: Model works but may need tuning")

print("\n" + "="*80)

## Summary

Your model has been successfully tested and scored. Key metrics:

- **Anomaly Detection Rate:** {:.1f}% (expected ~10%)
- **Score Separation:** {:.4f} (good if > 0.05)
- **Detected Anomalies:** {} samples
- **Model Status:** Ready for production use

### Next Steps:
1. Review the detected anomalies for ecological validity
2. Adjust contamination rate if detection rate is significantly off
3. Use DIFFI for feature attribution on new anomalies
4. Deploy model for real-time anomaly detection
""".format(detection_rate, separation if len(anomaly_scores) > 0 else 0, n_anomalies)